# Module 6 - Attention From Scratch

## What you'll build

This is the idea that makes transformers work: **self-attention**. The
MLP in Module 5 mashed the whole context into one flat vector. Attention
is smarter -- it lets every position look back at the earlier positions
and decide, on the fly, which ones are worth paying attention to.

You'll implement a single causal attention head in NumPy:

1. Project the input `X` (shape `T x C`) into three things via seeded
   weight matrices: **queries** `Q`, **keys** `K`, and **values** `V`.
2. Score every pair of positions: `scores = Q @ K.T / sqrt(head_size)`.
   The scaling keeps the numbers in a sane range.
3. Apply a **causal mask**: set the strict upper triangle to `-inf` so a
   position can only attend to itself and earlier positions (a model
   predicting the next token must not peek at the future).
4. `softmax` each row into attention weights, then mix the values:
   `out = weights @ V`.

Then wrap several heads into **`multi_head`**: run `n_head` independent
heads (each with its own seed), concatenate their outputs, and apply a
final seeded output projection. That's the whole attention mechanism --
the rest of a GPT is embeddings, MLPs, and LayerNorm around this core.

In [ ]:
import numpy as np

def self_attention(X, head_size, seed, return_weights=False):
    # TODO: T, C = X.shape; rng = np.random.default_rng(seed).
    #   - build Wq, Wk, Wv each (C x head_size) from the SAME rng
    #   - Q, K, V = X @ Wq, X @ Wk, X @ Wv
    #   - scores = Q @ K.T / sqrt(head_size)
    #   - causal mask: strict upper triangle -> -inf
    #   - weights = softmax(scores) row-wise; out = weights @ V
    #   - return (out, weights) if return_weights else out
    raise NotImplementedError

def multi_head(X, n_head, seed):
    # TODO: head_size = C // n_head; run n_head heads with seeds seed+i,
    #       concatenate to (T, C), then apply a seeded output proj Wo (C x C).
    raise NotImplementedError


## Reveal the reference solution

Run the hidden cell to load the reference `self_attention` and
`multi_head` (it redefines both with the complete implementation,
including the small `_softmax` helper they use).

In [ ]:
# Reference solution (single source of truth: reference/attention/attention.py)

"""Self-attention from scratch in NumPy (M6).

Implements a single causal self-attention head and a multi-head wrapper.
Everything is plain NumPy so students can follow each step:
project to Q/K/V, score, causally mask, softmax, then mix the values.
"""

import numpy as np


def _softmax(scores):
    """Row-wise softmax over the last axis (numerically stable)."""
    # Subtract the per-row max so exp() never overflows. Rows that are all
    # -inf (impossible here, every row keeps at least its own position)
    # would need extra care, but causal masking always leaves the diagonal.
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


def self_attention(X, head_size, seed, return_weights=False):
    """One causal self-attention head.

    X is (T, C). We build seeded Q/K/V projection matrices (C x head_size),
    compute scaled dot-product scores, apply a causal mask (each position may
    only attend to positions <= its own index), softmax, then weight the
    values. Returns (T, head_size), or (out, weights) if return_weights.
    """
    T, C = X.shape
    rng = np.random.default_rng(seed)

    # Seeded projection matrices. Drawing all three from the same rng keeps
    # the result fully determined by `seed`.
    Wq = rng.standard_normal((C, head_size))
    Wk = rng.standard_normal((C, head_size))
    Wv = rng.standard_normal((C, head_size))

    Q = X @ Wq            # (T, head_size)
    K = X @ Wk            # (T, head_size)
    V = X @ Wv            # (T, head_size)

    # Scaled dot-product attention scores: (T, T).
    scores = Q @ K.T / np.sqrt(head_size)

    # Causal mask: set the strict upper triangle to -inf BEFORE softmax so a
    # position can only attend to itself and earlier positions.
    mask = np.triu(np.ones((T, T), dtype=bool), k=1)
    scores = np.where(mask, -np.inf, scores)

    weights = _softmax(scores)   # (T, T), each row sums to 1
    out = weights @ V            # (T, head_size)

    if return_weights:
        return out, weights
    return out


def multi_head(X, n_head, seed):
    """Multi-head self-attention.

    X is (T, C) with C divisible by n_head. We run n_head independent heads
    (each of size C // n_head, with its own seed), concatenate their outputs
    back to (T, C), then apply a seeded output projection (C x C).
    """
    T, C = X.shape
    assert C % n_head == 0, "C must be divisible by n_head"
    head_size = C // n_head

    # Each head gets a distinct seed so they learn different projections.
    heads = [self_attention(X, head_size, seed + i) for i in range(n_head)]
    concat = np.concatenate(heads, axis=-1)   # (T, C)

    # Seeded output projection mixes information across heads.
    rng = np.random.default_rng(seed)
    Wo = rng.standard_normal((C, C))
    return concat @ Wo                         # (T, C)


## Check your work

The grader checks output shape, that attention weights sum to 1 per row,
**causality** (changing future inputs must not change earlier outputs),
and that `multi_head` preserves the channel dimension.

In [ ]:
import sys, os
# Make the repo root importable so `grader` / `reference` resolve.
_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)
from grader.checks.m6 import check_attention
from grader.core import run_checks

r = run_checks(check_attention(self_attention, multi_head))
print('PASS' if r.passed else 'FAIL', f'score={r.score:.2f}')
if not r.passed:
    print('failed checks:', r.failed_checks)
